# SVTRv2X: Full End-to-End Pipeline (Google Colab)
This notebook runs SVTRv2X Tiny training on Union14M-L-Filter with checkpoints saved directly to Google Drive.

### Step 1: Check GPU

In [ ]:
!nvidia-smi

### Step 2: Mount Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/svtrv2x_checkpoints/union14m_tiny', exist_ok=True)

### Step 3: Clone Repository

In [ ]:
%cd /content
!rm -rf AMSTR
!git clone https://github.com/GodishalaAshwith/AMSTR.git
%cd /content/AMSTR

### Step 4: Install Dependencies

In [ ]:
!pip install -q lmdb pyclipper pyyaml rapidfuzz tqdm huggingface_hub torchvision

### Step 5: Download Full Training & Evaluation Datasets

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="topdu/OpenOCR-Data",
    repo_type="dataset",
    allow_patterns=["Union14M-L-LMDB-Filtered/**", "evaluation/**"],
    local_dir="/content/AMSTR"
)

### Step 6: Verify Architecture & Modules (Run 1 & Run 2 Tier 1)

In [ ]:
!python tools/test_svtrv2x.py
!python tools/test_svtrv2x_tier1.py

### Step 7a: Launch Training — Run 1 (Baseline SVTRv2X Tiny)

In [ ]:
!python tools/train_rec.py --c configs/rec/svtrv2/svtrv2x_tiny_union14m.yml \
    -o Global.output_dir=/content/drive/MyDrive/svtrv2x_checkpoints/union14m_tiny \
       Global.save_epoch_step="[0, 1]" \
       Train.sampler.first_bs=64 \
       Train.loader.batch_size_per_card=64

### Step 7b: Launch Training — Run 2 (Tier 1 Architecture Injection: SE-Stem + 2D Pos Embed)

In [ ]:
!python tools/train_rec.py --c configs/rec/svtrv2/svtrv2x_tiny_union14m_run2_tier1.yml \
    -o Global.output_dir=/content/drive/MyDrive/svtrv2x_checkpoints/union14m_tiny_run2_tier1 \
       Global.save_epoch_step="[0, 1]" \
       Train.sampler.first_bs=64 \
       Train.loader.batch_size_per_card=64

### Step 8: Resume Training (If Disconnected)

In [ ]:
# Resume Run 2 (or change path to union14m_tiny for Run 1)
%cd /content/AMSTR
!python tools/train_rec.py --c configs/rec/svtrv2/svtrv2x_tiny_union14m_run2_tier1.yml \
    -o Global.checkpoints=/content/drive/MyDrive/svtrv2x_checkpoints/union14m_tiny_run2_tier1/latest.pth \
       Global.output_dir=/content/drive/MyDrive/svtrv2x_checkpoints/union14m_tiny_run2_tier1 \
       Train.sampler.first_bs=64 \
       Train.loader.batch_size_per_card=64

### Step 9: Standalone Evaluation on Benchmark Datasets

In [ ]:
# Evaluate Run 2 Tier 1 model
!python tools/eval_rec.py --c configs/rec/svtrv2/svtrv2x_tiny_union14m_run2_tier1.yml \
    -o Global.checkpoints=/content/drive/MyDrive/svtrv2x_checkpoints/union14m_tiny_run2_tier1/best.pth